In [3]:
import pandas as pd
import numpy as np

DATE_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df = pd.read_csv("funnel_df.csv", parse_dates=DATE_COLS)

# 구매 월
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')

# 고객별 첫 구매월
first_purchase = (
    df.groupby('customer_unique_id')['order_purchase_timestamp']
    .min()
    .dt.to_period('M')
    .rename('first_purchase_month')
)
df = df.merge(first_purchase, on='customer_unique_id', how='left')

# 코호트 인덱스
df['cohort_index'] = (
    df['order_month'] - df['first_purchase_month']
).apply(lambda x: x.n if pd.notna(x) else np.nan).astype('Int64')

# 첫 구매 시 지연 경험 여부
first_order_delay = (
    df.sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id', as_index=False)
    .agg(first_order_is_delayed=('is_delayed', 'first'))
)
df = df.merge(first_order_delay, on='customer_unique_id', how='left')

# 재구매 여부
df['is_repurchase'] = (df['cohort_index'] > 0).astype(int)

# 저장
df.to_csv("cohort_df.csv", index=False)

print('shape:', df.shape)
print('추가된 컬럼: order_month, first_purchase_month, cohort_index, first_order_is_delayed, is_repurchase')
print('저장 완료: cohort_df.csv')

shape: (99441, 23)
추가된 컬럼: order_month, first_purchase_month, cohort_index, first_order_is_delayed, is_repurchase
저장 완료: cohort_df.csv
